# Kontrollitud tunnusekatsed ja XAI lisaväärtus

RQ2 keskne võrdlus on veapõhine valik versus XAI-põhine valik. Eelmääratud üldine tunnusevalik on kolmas kontroll. Iga meetod saab samu kandidaate ja kaks katset kummaski juhtumiuuringus; hüperparameetreid ei optimeerita.

In [ ]:
from pathlib import Path
from functools import partial
import sys
from IPython.display import Markdown, display

ROOT = Path.cwd() if (Path.cwd() / "run_pipeline.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.common import latest_run, read_json
from src.reporting import show_table, show_run

RUN = latest_run()
notes = read_json(RUN / "reports/interpretations.json")
table = partial(show_table, RUN, limit=30)
show_run(RUN)

## Enne tulemusi fikseeritud otsused

Reeglite täpne valem on eksperimendiprotokollis. XAI paketis kasutatakse SHAP-i, PFI-d, analooge ja kaarte; sellest võrdlusest ei saa eraldada üksiku komponendi mõju.

In [ ]:
table(
    "refinements/decision_evidence.csv",
    columns=[
        "case",
        "group",
        "residual_score",
        "bridge_pfi",
        "matched_shap_contrast",
        "card_vote",
        "historical_association",
        "xai_score",
    ],
    rename={
        "residual_score": "Veapõhine skoor",
        "bridge_pfi": "Seotud baastunnuste PFI",
        "matched_shap_contrast": "Sobitatud SHAP-erinevus",
        "card_vote": "Kaartide soovituse osakaal",
        "historical_association": "Ajalooline seos",
        "xai_score": "XAI valikuskoor",
    },
)
display(
    Markdown(
        "Tabel näitab diagnoosiaasta tõenditest arvutatud skoorikomponente. Kaalud ei ole tulemustele sobitatud. Skoor on kontrollitav heuristika; selle kõrge väärtus ei tõenda tunnuse lisamise mõju."
    )
)
frozen = read_json(RUN / "refinements/frozen_decisions.json")
display(
    Markdown(
        "Otsused fikseeriti ajal `"
        + frozen["frozen_at_utc"]
        + "` UTC, enne valikuaasta tulemusi. Räsid on manifestis."
    )
)

In [ ]:
table(
    "refinements/all_trials.csv",
    columns=[
        "method",
        "case",
        "group",
        "MAE",
        "RMSE",
        "bias",
        "delta_MAE_vs_baseline",
        "hypothesis_supported_selection",
        "target_scenario_MAE",
        "target_scenario_delta_MAE",
        "quarter_MAE_std",
        "n_features",
        "tree_nodes",
    ],
    rename={
        "delta_MAE_vs_baseline": "MAE muutus baasist",
        "hypothesis_supported_selection": "Valideerimise MAE alla baasi",
    },
)
display(
    Markdown(
        notes["refinements"]
        + " Toetuse märgend tähendab üksnes madalamat üldist MAE-d; see ei kinnita mehhanismi ega statistilist paremust. Temperatuuri juhtumi sihtstsenaarium ühendab kuuma ja külma, muutuse juhtum kiire kasvu ja languse. Täielikud katsepõhised stsenaariumid on selection_scenarios.csv failis. Negatiivne MAE muutus tähendab paranemist."
    )
)
table("refinements/selected.csv", columns=["method", "case", "group", "MAE", "RMSE"])
display(
    Markdown(
        "Valitud rühm minimeerib meetodi kahe katse 2019 MAE-d. Üldise MAE põhjal valimine võib jätta stsenaariumipõhise hüpoteesi toetuseta; lõpphindamine toimub pärast selle valiku fikseerimist."
    )
)

In [ ]:
table(
    "refinements/seed_stability.csv",
    columns=["method", "case", "group", "seed", "MAE", "RMSE"],
    rename={"seed": "Juhuseeme"},
    limit=30,
)
display(
    Markdown(
        "Seemnete kontroll mõõdab sobituse tundlikkust fikseeritud tunnuserühmadel ja samal 2019 perioodil. Deterministliku GBR korral võib erinevus olla null. Ajast sõltuva üldistamise kontrolliks kasutatakse lisaks kvartalite MAE hajuvust ja eraldi lõpphindamist."
    )
)

## Ühe katse eelarve järelanalüüs

See lisakatse kasutab algset kandidaatide järjestust. Eelarve mõju uurimise otsus tehti pärast põhikatse tulemusi; tulemused ei ole uus sõltumatu kinnitus.

In [ ]:
if (RUN / "refinements/budget_metrics.csv").is_file():
    table(
        "refinements/budget_metrics.csv",
        columns=[
            "year",
            "method",
            "case",
            "budget",
            "MAE",
            "RMSE",
            "MASE",
            "scenario_n",
            "scenario_MAE",
        ],
        rename={
            "year": "Aasta",
            "budget": "Katseid juhtumi kohta",
            "scenario_n": "Olukorra tunde",
            "scenario_MAE": "Olukorra MAE",
        },
    )
    table(
        "evaluation/budget_intervals.csv",
        columns=["case", "scope", "difference", "adjusted_low", "adjusted_high", "block_hours"],
        rename={
            "scope": "Hinnatav osa",
            "difference": "XAI miinus veapõhise valiku MAE",
            "adjusted_low": "Vahemiku alampiir",
            "adjusted_high": "Vahemiku ülempiir",
        },
    )
else:
    display(Markdown("Selle käivituse katsete arvu analüüsi detailid puuduvad. Koondtulemused on põhiaruandes; detailide arvutamiseks käivita `run_pipeline.py`."))


## Abimudel ja prognoosivea parandamine

Võrdleme algse abimudeli prognoosi ühendamist ja varasemate vigade parandamist. Variant valitakse 2018.–2019. aasta üldise ja kiire kasvu MAE ning RMSE järgi. 2020. aasta oli juba varem vaadatud; selle tulemus on uuriv. Kõik variandid on ühes tabelis, lõppprognoosid ühes failis. SHAP-i lisakasu selles katses ei mõõdeta.

In [ ]:
from tools.build_report import specialist_section

display(Markdown(specialist_section(RUN)))

## Paranduskasu XAI-tagasiside ja Eesti ülekandekatse

Uus katseseeria kasutab varem realiseerunud paranduskasu Shapley-jaotust järgmiste paranduste juhtimiseks. Koosmõjuta ja ühe üldkaaluga tagasiside on eraldi kontrollid. Panama 2020 oli varem nähtud; Eesti 2024 puhul kasutati Panama arenduses fikseeritud seadeid. Allolev lahter loeb olemasolevaid tulemusi ega treeni mudeleid. Täpne meetod, uudsusotsing, piirangud ja ebaõnnestunud variandid on põhiaruandes ning katsete koondis.

In [ ]:
from src.common import read_artifact_csv
from IPython.display import Image

study = read_artifact_csv(ROOT / 'reports/trials.csv')
study = study[study.experiment.isin(['attribution_refinement', 'attribution_transfer'])]
display(study[study.stage == 'final'].pivot_table(index=['experiment', 'model'], columns='scope', values='MAE'))
for name in ('attribution_panama.png', 'attribution_estonia.png'):
    path = ROOT / 'reports' / name
    if path.exists():
        display(Image(filename=str(path)))